# 02 -- Team Batting Ingestion

Fetch team-level batting statistics from FanGraphs for seasons 2015-2024.

**Requirement:** DATA-02

**Source:** FanGraphs via `pybaseball.team_batting()`

**Key columns:** Team, G, PA, HR, R, RBI, SB, BB%, K%, ISO, BABIP, AVG, OBP, SLG, wOBA, wRC+, WAR

**Note:** 2020 season is flagged with `is_shortened_season=True` (60-game COVID season).

In [ ]:
# If editable install is not set up, uncomment the next line:
# import sys; sys.path.insert(0, "..")

from src.data.team_batting import fetch_team_batting
from src.data.cache import load_manifest

In [ ]:
# Fetch team batting stats for all 10 backtest seasons
seasons = range(2015, 2025)
dfs = {}
for season in seasons:
    print(f"Fetching {season}...")
    dfs[season] = fetch_team_batting(season)
    print(f"  {len(dfs[season])} teams")

In [ ]:
# Summary: show key batting metrics from 2023
df_sample = dfs[2023]
print(f"Columns: {list(df_sample.columns)}")
print(f"\nKey batting metrics (2023):")
display(df_sample[["Team", "G", "PA", "AVG", "OBP", "SLG", "wOBA", "wRC+"]].head(10))

print(f"\nwOBA range: {df_sample['wOBA'].min():.3f} - {df_sample['wOBA'].max():.3f}")
print(f"OPS range: {(df_sample['OBP'] + df_sample['SLG']).min():.3f} - {(df_sample['OBP'] + df_sample['SLG']).max():.3f}")

In [ ]:
# Coverage validation: verify 30 teams per season
for season, df in dfs.items():
    n_teams = len(df)
    short_flag = " (shortened)" if df["is_shortened_season"].iloc[0] else ""
    flag = "OK" if n_teams >= 30 else "LOW"
    print(f"{season}: {n_teams} teams [{flag}]{short_flag}")

# Specifically verify 2020 short-season flag
print(f"\n2020 is_shortened_season: {dfs[2020]['is_shortened_season'].iloc[0]}")
print(f"2020 season_games: {dfs[2020]['season_games'].iloc[0]}")

In [ ]:
# Cache check
manifest = load_manifest()
batting_keys = {k: v for k, v in manifest.items() if k.startswith("team_batting_")}
print(f"Cached team batting files: {len(batting_keys)}")
for k, v in sorted(batting_keys.items()):
    print(f"  {k}: {v['row_count']} rows, fetched {v['fetch_date'][:10]}")